In [3]:
import argparse
import os

import yaml

import autoslo.utils.paths as pu
from autoslo.featurization.iconq_query_featurizer import IconqQueryFeaturizer
from autoslo.models.cache_model import CacheModel
from autoslo.models.iconq_model import (
    IconqModel,
    IconqModelInitConfig,
    NNModelTrainConfig,
)
from autoslo.models.iconq_model_trainer import iconq_model_trainer
from autoslo.models.stage_model import StageModel
from autoslo.models.xgboost_model import XGBoostModel

from datetime import datetime

from autoslo.workload_execution.trace import Trace
import pandas as pd


In [4]:
pct_heavy_options = [0, 10, 25, 50]
mean_interarrival_options = [10, 30, 60, 120]
rpus = [4, 8, 16, 32]

latencies_all = []
run_ids = set()


# Most recent chunk runs.
for i, pct_heavy in enumerate(pct_heavy_options):
    for j, mean_interarrival in enumerate(mean_interarrival_options):
        for k, rpu in enumerate(rpus):
            this_workload_run_ids = pu.RunLocator.get_run_ids(
                schema_name="ext_tpcds1000",
                workload_name=f"tpcds_99templates_{pct_heavy:02d}pctheavy_{mean_interarrival}meaninterarrivals",
                blueprint_name=f"single_{rpu}",
            )
            run_id = max(this_workload_run_ids)
            run_ids.add(run_id)

            trace = Trace(run_id)
            latencies = trace.latencies_s
            latencies_all.append(latencies)

assert len(run_ids) == len(pct_heavy_options) * len(mean_interarrival_options) * len(rpus)

latencies_unified = pd.concat(latencies_all)

In [6]:
latencies_unified.describe()

count    9020.000000
mean      384.237023
std       872.136692
min         0.098077
25%         3.140796
50%        20.204583
75%       165.159390
max      3600.041982
Name: elapsed_time, dtype: float64

### The minimum latency we see is 98 ms so it's reasonable to enforce a lower bound of say 1 ms on all latencies (just in case) and then use the log instead of the log1p.